In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

In [2]:
EPOCHS = 32
BATCH_SIZE = 256
INPUT_FILE_NAME = '../datasets/frankenstein.txt'
WINDOW_LENGTH = 40
WINDOW_STEP = 3

LESS_EPOCHS = 15
MORE_BATCH = 512

In [3]:
# Open the input file.
file = open(INPUT_FILE_NAME, 'r', encoding='utf-8')
text = file.read()
file.close()



In [4]:
# Train Just Chapter 1

# # remove gutenberg boilerplate
# start_marker = "CHAPTER I."
# # end_marker   = "THE END."
# # Working with smaller dataset
# end_marker   = "CHAPTER II."

# start = text.find(start_marker)
# end   = text.find(end_marker)

# if start != -1 and end != -1:
#     text = text[start+len(start_marker):end].strip()   #.strip() removes leading and trailing whitespace from a string
# else:
#     text = text

In [5]:
text

'The Project Gutenberg eBook of Frankenstein; Or, The Modern Prometheus\n    \nThis eBook is for the use of anyone anywhere in the United States and\nmost other parts of the world at no cost and with almost no restrictions\nwhatsoever. You may copy it, give it away or re-use it under the terms\nof the Project Gutenberg License included with this eBook or online\nat www.gutenberg.org. If you are not located in the United States,\nyou will have to check the laws of the country where you are located\nbefore using this eBook.\n\nTitle: Frankenstein; Or, The Modern Prometheus\n\nAuthor: Mary Wollstonecraft Shelley\n\n\n        \nRelease date: March 13, 2013 [eBook #42324]\n                Most recently updated: October 23, 2024\n\nLanguage: English\n\nOriginal publication: United Kingdom: H. Colburn and R. Bentley, 1831\n\nOther information and formats: www.gutenberg.org/ebooks/42324\n\nCredits: Produced by Greg Weeks, Mary Meehan and the Online\n        Distributed Proofreading Team at htt

In [6]:

# Make lowercase and remove newline and extra spaces.
text = text.lower()
text = text.replace('\n', ' ')
text = text.replace(' ', ' ')
text[:500]

'the project gutenberg ebook of frankenstein; or, the modern prometheus      this ebook is for the use of anyone anywhere in the united states and most other parts of the world at no cost and with almost no restrictions whatsoever. you may copy it, give it away or re-use it under the terms of the project gutenberg license included with this ebook or online at www.gutenberg.org. if you are not located in the united states, you will have to check the laws of the country where you are located before'

In [7]:
len(text)

457713

In [8]:
print(set(text))

{'.', 'æ', 'i', '*', 'f', 'e', '?', '[', '‘', '%', ',', '-', '—', '2', 'z', ' ', 'ô', '8', '$', '!', 'h', 's', '_', '4', 'w', 'è', '3', 'm', '’', '7', 'k', '+', 'r', 'b', 'é', 'j', 'l', 'd', '5', '6', 'n', 'y', ':', '#', 'o', '™', '”', 'c', 'p', 'v', '“', 'q', "'", '0', 'g', 'x', ')', '(', 'u', ']', '9', '1', '•', 'ê', '&', ';', '/', '"', 'a', 't'}


In [9]:
# Encode characters as indices.
unique_chars = list(set(text))
print(unique_chars)

['.', 'æ', 'i', '*', 'f', 'e', '?', '[', '‘', '%', ',', '-', '—', '2', 'z', ' ', 'ô', '8', '$', '!', 'h', 's', '_', '4', 'w', 'è', '3', 'm', '’', '7', 'k', '+', 'r', 'b', 'é', 'j', 'l', 'd', '5', '6', 'n', 'y', ':', '#', 'o', '™', '”', 'c', 'p', 'v', '“', 'q', "'", '0', 'g', 'x', ')', '(', 'u', ']', '9', '1', '•', 'ê', '&', ';', '/', '"', 'a', 't']


In [10]:

char_to_index = dict((ch, index) for index, ch in enumerate(unique_chars))
print(char_to_index)

{'.': 0, 'æ': 1, 'i': 2, '*': 3, 'f': 4, 'e': 5, '?': 6, '[': 7, '‘': 8, '%': 9, ',': 10, '-': 11, '—': 12, '2': 13, 'z': 14, ' ': 15, 'ô': 16, '8': 17, '$': 18, '!': 19, 'h': 20, 's': 21, '_': 22, '4': 23, 'w': 24, 'è': 25, '3': 26, 'm': 27, '’': 28, '7': 29, 'k': 30, '+': 31, 'r': 32, 'b': 33, 'é': 34, 'j': 35, 'l': 36, 'd': 37, '5': 38, '6': 39, 'n': 40, 'y': 41, ':': 42, '#': 43, 'o': 44, '™': 45, '”': 46, 'c': 47, 'p': 48, 'v': 49, '“': 50, 'q': 51, "'": 52, '0': 53, 'g': 54, 'x': 55, ')': 56, '(': 57, 'u': 58, ']': 59, '9': 60, '1': 61, '•': 62, 'ê': 63, '&': 64, ';': 65, '/': 66, '"': 67, 'a': 68, 't': 69}


In [11]:

index_to_char = dict((index, ch) for index, ch in enumerate(unique_chars))
print(index_to_char)

{0: '.', 1: 'æ', 2: 'i', 3: '*', 4: 'f', 5: 'e', 6: '?', 7: '[', 8: '‘', 9: '%', 10: ',', 11: '-', 12: '—', 13: '2', 14: 'z', 15: ' ', 16: 'ô', 17: '8', 18: '$', 19: '!', 20: 'h', 21: 's', 22: '_', 23: '4', 24: 'w', 25: 'è', 26: '3', 27: 'm', 28: '’', 29: '7', 30: 'k', 31: '+', 32: 'r', 33: 'b', 34: 'é', 35: 'j', 36: 'l', 37: 'd', 38: '5', 39: '6', 40: 'n', 41: 'y', 42: ':', 43: '#', 44: 'o', 45: '™', 46: '”', 47: 'c', 48: 'p', 49: 'v', 50: '“', 51: 'q', 52: "'", 53: '0', 54: 'g', 55: 'x', 56: ')', 57: '(', 58: 'u', 59: ']', 60: '9', 61: '1', 62: '•', 63: 'ê', 64: '&', 65: ';', 66: '/', 67: '"', 68: 'a', 69: 't'}


In [12]:
encoding_width = len(char_to_index)

In [13]:
encoding_width

70

In [14]:
# Create training examples.
fragments = []
targets = []

for i in range(0, len(text) - WINDOW_LENGTH, WINDOW_STEP):
    fragments.append(text[i: i + WINDOW_LENGTH])
    targets.append(text[i + WINDOW_LENGTH])

In [15]:
print(fragments)

['the project gutenberg ebook of frankenst', ' project gutenberg ebook of frankenstein', 'oject gutenberg ebook of frankenstein; o', 'ct gutenberg ebook of frankenstein; or, ', 'gutenberg ebook of frankenstein; or, the', 'enberg ebook of frankenstein; or, the mo', 'erg ebook of frankenstein; or, the moder', ' ebook of frankenstein; or, the modern p', 'ook of frankenstein; or, the modern prom', ' of frankenstein; or, the modern prometh', ' frankenstein; or, the modern prometheus', 'ankenstein; or, the modern prometheus   ', 'enstein; or, the modern prometheus      ', 'tein; or, the modern prometheus      thi', 'n; or, the modern prometheus      this e', 'or, the modern prometheus      this eboo', ' the modern prometheus      this ebook i', 'e modern prometheus      this ebook is f', 'odern prometheus      this ebook is for ', 'rn prometheus      this ebook is for the', 'prometheus      this ebook is for the us', 'metheus      this ebook is for the use o', 'heus      this ebook is for th

In [16]:
len(fragments)

152558

In [17]:
# Convert to one-hot encoded training data.
X = np.zeros((len(fragments), WINDOW_LENGTH, encoding_width), dtype=np.float32)
y = np.zeros((len(fragments), encoding_width), dtype=np.float32)
for i, fragment in enumerate(fragments):
    for j, char in enumerate(fragment):
        X[i, j, char_to_index[char]] = 1
    target_char = targets[i]
    y[i, char_to_index[target_char]] = 1

In [18]:
print(tf.config.list_physical_devices("GPU"))

[]


In [19]:
# Build and train model.
model = Sequential()
# model.add(LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2,input_shape=(None, encoding_width)))
# model.add(LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2,input_shape=(None, encoding_width)))
# model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
# Remove recurrent dropout for better performance on GPU
model.add(LSTM(128, return_sequences=True, dropout=0.2,input_shape=(None, encoding_width)))
model.add(LSTM(128, return_sequences=True, dropout=0.2,input_shape=(None, encoding_width)))
model.add(LSTM(128, dropout=0.2))
model.add(Dense(encoding_width, activation='softmax'))
model.compile(loss='categorical_crossentropy',optimizer='adam')
model.summary()

/Users/shubham.kumar.3/Desktop/Deep-Learning/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, None, 128)      │       101,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, None, 128)      │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 70)             │         9,030 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 374,086 (1.43 MB)

 Trainable params: 374,086 (1.43 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:

history = model.fit(X, y, validation_split=0.05,batch_size=BATCH_SIZE,epochs=EPOCHS, verbose=2,shuffle=True)

Epoch 1/32
567/567 - 87s - 153ms/step - loss: 2.9081 - val_loss: 2.7358
Epoch 2/32
567/567 - 86s - 151ms/step - loss: 2.5273 - val_loss: 2.5491
Epoch 3/32
567/567 - 83s - 147ms/step - loss: 2.4094 - val_loss: 2.4857
Epoch 4/32
567/567 - 82s - 144ms/step - loss: 2.3378 - val_loss: 2.4018
Epoch 5/32
567/567 - 80s - 141ms/step - loss: 2.2830 - val_loss: 2.3611
Epoch 6/32
567/567 - 81s - 143ms/step - loss: 2.2414 - val_loss: 2.3275
Epoch 7/32
567/567 - 81s - 143ms/step - loss: 2.2059 - val_loss: 2.3043
Epoch 8/32
567/567 - 78s - 138ms/step - loss: 2.1729 - val_loss: 2.2805
Epoch 9/32
567/567 - 78s - 138ms/step - loss: 2.1449 - val_loss: 2.2387
Epoch 10/32
567/567 - 77s - 136ms/step - loss: 2.1177 - val_loss: 2.2272
Epoch 11/32
567/567 - 75s - 132ms/step - loss: 2.0999 - val_loss: 2.2141
Epoch 12/32
567/567 - 72s - 127ms/step - loss: 2.0782 - val_loss: 2.1931
Epoch 13/32
567/567 - 76s - 134ms/step - loss: 2.0582 - val_loss: 2.1837
Epoch 14/32
567/567 - 72s - 127ms/step - loss: 2.0443 - val_

In [21]:
letters = 'the boy '
one_hots = []
for i, char in enumerate(letters):
    x = np.zeros(encoding_width)
    x[char_to_index[char]] = 1
    one_hots.append(x)

new = np.array(one_hots).reshape(1, len(letters), encoding_width)

In [22]:
new.shape

(1, 8, 70)

In [23]:
y_predict = model.predict(new, verbose=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step


In [24]:
y_predict

array([[3.20586783e-04, 4.18968602e-06, 2.74949614e-02, 8.91197669e-06,
        3.62010673e-02, 2.71959025e-02, 1.31425913e-05, 1.66313157e-06,
        1.58470026e-09, 8.91767160e-10, 5.65311871e-04, 2.98747967e-04,
        1.98988603e-09, 8.55983526e-05, 4.93767620e-05, 2.74396199e-03,
        9.65222480e-09, 1.14236535e-07, 2.35849140e-09, 2.70287419e-05,
        7.35841617e-02, 5.64838760e-02, 4.87253565e-04, 6.47832258e-06,
        7.07835555e-02, 1.85213334e-09, 1.52285537e-07, 8.02284330e-02,
        1.14356313e-09, 2.03408763e-05, 4.70310031e-03, 1.67723158e-09,
        3.70835662e-02, 3.20822448e-02, 1.53078297e-08, 2.54147127e-03,
        3.29209231e-02, 3.60533185e-02, 5.26167014e-07, 3.26761437e-06,
        3.27281058e-02, 6.81310473e-03, 1.78831124e-05, 1.47728063e-09,
        5.77677190e-02, 1.11101239e-09, 1.77286841e-09, 3.35226618e-02,
        2.61748638e-02, 1.02277305e-02, 1.41801215e-09, 1.05423969e-03,
        3.95177631e-05, 2.78901151e-08, 5.34798997e-03, 3.555734

In [25]:
np.argmax(y_predict)

np.int64(68)

In [26]:
index_to_char[np.argmax(y_predict)]

'a'

In [27]:
# Number of characters to generate
NUM_PREDICTIONS = 12

# Seed text
seed_text = "the boy who "
generated = seed_text

# Prepare one-hot input from the seed text
one_hots = []
for char in seed_text:
    x = np.zeros(encoding_width)
    x[char_to_index[char]] = 1
    one_hots.append(x)

input_seq = np.array(one_hots).reshape(1, len(seed_text), encoding_width)

# Generate next NUM_PREDICTIONS characters
for i in range(NUM_PREDICTIONS):
    # Predict probabilities for next char
    y_predict = model.predict(input_seq, verbose=0)
    next_index = np.argmax(y_predict)
    next_char = index_to_char[next_index]

    print(f"Predicted letter {i+1}: {next_char}")
    generated += next_char

    # Update input sequence: append new char & drop oldest if longer than WINDOW_LENGTH
    next_one_hot = np.zeros((1, 1, encoding_width))
    next_one_hot[0, 0, next_index] = 1
    input_seq = np.concatenate([input_seq, next_one_hot], axis=1)

    if input_seq.shape[1] > WINDOW_LENGTH:
        input_seq = input_seq[:, 1:, :]   # keep only last WINDOW_LENGTH chars

print("\nGenerated text:\n", generated)

Predicted letter 1: h
Predicted letter 2: a
Predicted letter 3: d
Predicted letter 4:  
Predicted letter 5: t
Predicted letter 6: h
Predicted letter 7: e
Predicted letter 8:  
Predicted letter 9: s
Predicted letter 10: a
Predicted letter 11: m
Predicted letter 12: e

Generated text:
 the boy who had the same


In [28]:
## save model for reuse
model.save("../models/language_model_LSTM.keras")

In [29]:
## load saved model
loaded_model = keras.models.load_model("../models/language_model_LSTM.keras")

In [30]:
## predict using saved model
y_test = loaded_model(new)

In [31]:
## convert index to character
index_to_char[np.argmax(y_test)]

'a'

In [32]:
# Number of characters to generate
NUM_PREDICTIONS = 15
encoding_width=loaded_model.input_shape[-1] # use model's expected input width

# Seed text
seed_text = "the gulf "
generated = seed_text

# Prepare one-hot input from the seed text
one_hots = []
for char in seed_text:
    x = np.zeros(encoding_width)
    x[char_to_index[char]] = 1
    one_hots.append(x)

input_seq = np.array(one_hots).reshape(1, len(seed_text), encoding_width)

# Generate next NUM_PREDICTIONS characters
for i in range(NUM_PREDICTIONS):
    # Predict probabilities for next char
    y_predict = loaded_model.predict(input_seq, verbose=0)
    next_index = np.argmax(y_predict)
    next_char = index_to_char[next_index]

    print(f"Predicted letter {i+1}: {next_char}")
    generated += next_char

    # Update input sequence: append new char & drop oldest if longer than WINDOW_LENGTH
    next_one_hot = np.zeros((1, 1, encoding_width))
    next_one_hot[0, 0, next_index] = 1
    input_seq = np.concatenate([input_seq, next_one_hot], axis=1)

    if input_seq.shape[1] > WINDOW_LENGTH:
        input_seq = input_seq[:, 1:, :]   # keep only last WINDOW_LENGTH chars

print("\nGenerated text:\n", generated)

Predicted letter 1: a
Predicted letter 2: n
Predicted letter 3: d
Predicted letter 4:  
Predicted letter 5: a
Predicted letter 6: n
Predicted letter 7: d
Predicted letter 8:  
Predicted letter 9: a
Predicted letter 10: n
Predicted letter 11: d
Predicted letter 12:  
Predicted letter 13: a
Predicted letter 14: n
Predicted letter 15: d

Generated text:
 the gulf and and and and
